# Data preparation for Proof-of-Concept study (Figure 2)

This notebook prepares data for Proof-of-Concept study.

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import os.path as op
from tqdm import tqdm
sys.path.append("..")
from Bio.Seq import Seq
from Bio import SeqIO, motifs
import src.swiper_cli as swiper
from src.convergence_test import get_consensus
from src.passive_learning import load_PPDV_data, get_Xy

First we setup the filenames for all input and output data:

In [2]:
PATH = "../data/external/41586_2015_BFnature14592_MOESM46_ESM.xlsx"
GS_PATH = "../data/processed/GS_6nt.fasta"
LIBRARIES = ["library1", "library2"]
SETUPS=["complete_pool", "golden_set"]
OUTDIR = "../data/interim/fig2/fillings/LIBRARY/SETUP/"
CONSENSI_FILE = "../data/processed/consensi.json"

Create the required directories:

In [3]:
for library in LIBRARIES:
    for setup in SETUPS:
        os.makedirs(OUTDIR.replace("LIBRARY", library).replace("SETUP", setup), exist_ok=True)

Load the Kleinstiver, et al, 2015 [29] data:

In [4]:
library1 = load_PPDV_data(excel_file=PATH, sheet_name="library 1")
library2 = load_PPDV_data(excel_file=PATH, sheet_name="library 2")
orthologs = list(filter(lambda x: "PPDV" in x, library1.columns))
GS_6NT = np.array(
    [str(a.seq) for a in SeqIO.parse(GS_PATH, "fasta")]
)
library1_GS = library1.loc[np.isin(library1["6nt PAM"].values, GS_6NT)]
library2_GS = library2.loc[np.isin(library2["6nt PAM"].values, GS_6NT)]

/home/drlapki/micromamba/envs/ml-generic/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/home/drlapki/micromamba/envs/ml-generic/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


To compare our PAM predictions to the ones that can be determined by the depletion screen, we did not use the aforementioned PAM sequences given in the text of
Kleinstiver et al., 2015 study [29]. Instead, we used the degenerate consensus sequences of
all cleaved PAM candidates from the dataset. This decision is motivated by the
fact, that the PAM sequences established in the Kleinstiver et al., 2015 paper are
determined after additional experiments in human cells, which are outside the scope
for the current research. In the Proof-of-Concept study we are interested in recovering
the PAM from depletion screen data while using a small strategically selected subset
of it, not in prediction of outcome for a different, albeit related, biological experiment.

In [5]:
consensi = {
    "library1": {},
    "library2": {}
}

In [7]:
for S,SETUP in zip(SETUPS, [[library1, library2], [library1_GS, library2_GS]]):
    for L,lib in zip(LIBRARIES, SETUP):
        for a in orthologs[1:]:
            _, y = get_Xy(lib, a)
            current_filling = pd.DataFrame(
                {"PAM": lib["6nt PAM"].values, "Outcome": ["-" if a == 0 else "+" for a in y]}
            )
            current_filling.to_csv(
                op.join(OUTDIR.replace("SETUP", S).replace("LIBRARY", L), a.replace(" ", "_")+".tsv"), sep="\t"
            )
            consensi[L][a] = get_consensus(current_filling.loc[current_filling["Outcome"] == "+"]["PAM"].values)
            print(S, L, a, np.unique(current_filling["Outcome"].values, return_counts=True))

complete_pool library1 wt SpCas9 PPDV (array(['+', '-'], dtype=object), array([ 516, 3580]))
complete_pool library1 SpCas9 VQR PPDV (array(['+', '-'], dtype=object), array([ 538, 3558]))
complete_pool library1 SpCas9 EQR PPDV (array(['+', '-'], dtype=object), array([ 263, 3833]))
complete_pool library1 SpCas9 VRER PPDV (array(['+', '-'], dtype=object), array([  78, 4018]))
complete_pool library1 SpCas9 D1135E PPDV (array(['+', '-'], dtype=object), array([ 327, 3769]))
complete_pool library1 St1Cas9 20nt PPDV (array(['+', '-'], dtype=object), array([ 253, 3843]))
complete_pool library1 St1Cas9 21nt PPDV (array(['+', '-'], dtype=object), array([ 137, 3959]))
complete_pool library1 SaCas9 21nt PPDV (array(['+', '-'], dtype=object), array([  37, 4059]))
complete_pool library1 SaCas9 23nt PPDV (array(['+', '-'], dtype=object), array([ 127, 3969]))
complete_pool library2 wt SpCas9 PPDV (array(['+', '-'], dtype=object), array([ 498, 3598]))
complete_pool library2 SpCas9 VQR PPDV (array(['+', 

In [8]:
consensi

{'library1': {'wt SpCas9 PPDV': 'SGGGMN',
  'SpCas9 VQR PPDV': 'NGNGSN',
  'SpCas9 EQR PPDV': 'SGAKGM',
  'SpCas9 VRER PPDV': 'DGCGGC',
  'SpCas9 D1135E PPDV': 'SGGGNN',
  'St1Cas9 20nt PPDV': 'NNANAA',
  'St1Cas9 21nt PPDV': 'SNRGAA',
  'SaCas9 21nt PPDV': 'SMGART',
  'SaCas9 23nt PPDV': 'SMGRRW'},
 'library2': {'wt SpCas9 PPDV': 'SGGGNN',
  'SpCas9 VQR PPDV': 'NGNGNN',
  'SpCas9 EQR PPDV': 'NGAKGN',
  'SpCas9 VRER PPDV': 'DGCGGC',
  'SpCas9 D1135E PPDV': 'SGGGNN',
  'St1Cas9 20nt PPDV': 'CWARAA',
  'St1Cas9 21nt PPDV': 'MNRGAA',
  'SaCas9 21nt PPDV': 'SNGRNN',
  'SaCas9 23nt PPDV': 'SNGRNN'}}

In [9]:
with open(CONSENSI_FILE, "w") as oh:
    oh.write(json.dumps(consensi))

# References

29. Kleinstiver, B. P. et al. Engineered crispr-cas9 nucleases with altered pam specificities. Nature 523, 481–485 (2015).